# 3. Method clustering

Part of the **[Fig. 1 chapter](fig1.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
from glob import glob

import anndata
import matplotlib as mpl
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import seaborn as sns

from ALLCools.clustering import *
from ALLCools.integration.seurat_class import SeuratIntegration
from ALLCools.plot import *
from ALLCools.mcds import MCDS

from sklearn.metrics import pairwise_distances, roc_auc_score, average_precision_score
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

mpl.style.use("default")
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = "Helvetica"

import warnings
warnings.filterwarnings("ignore")


In [ ]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [ ]:
indir = f'{REF_ROOT}/wmb/'
outdir = f'{ENTEX_ROOT}/analysis/method/'


In [ ]:
meta = pd.read_csv(f'{indir}CEMBA.mC.Metadata.csv.gz', header=0, index_col=0)
meta

In [ ]:
L2_color = pd.read_csv(f'{indir}AIT21_SubClassColor.csv', header=None, index_col=0)[1]
L2_color

In [ ]:
gene_meta_path = f'{REF_ROOT}/mm10/gencode/biccn/modified_gencode.vM23.primary_assembly.annotation.gene.flat.tsv.gz'
black_list_path = f'{REF_ROOT}/blacklist/mm10-blacklist.v2.bed.gz'

chrom_to_remove = ['chrL', 'chrM', 'chrX', 'chrY']


In [ ]:
obs_dim = 'cell'
var_dim = 'chrom100k'


In [ ]:
mcds = MCDS.open(f'{indir}CEMBA.snmC.mcds', var_dim=var_dim)
mcds

In [ ]:
mcds = mcds.sel({obs_dim: mcds.get_index('cell').intersection(meta.index)})
mcds = mcds.remove_chromosome(exclude_chromosome=chrom_to_remove, var_dim=var_dim)
mcds = mcds.remove_black_list_region(black_list_path=black_list_path, f=0.5)


In [ ]:
feature_cov_mean = mcds['chrom100k_cov_mean'].to_pandas()
feature_cov_mean

In [ ]:
# mc_type = 'CHN'
# feature_cov_mean = mcds[f'{var_dim}_da'].sel(mc_type=mc_type, count_type='cov').mean(dim=obs_dim).to_pandas()
fig, ax = plt.subplots(figsize=(4,2), dpi=300)
sns.histplot(feature_cov_mean, bins=100, ax=ax)


In [ ]:
use_features = feature_cov_mean[feature_cov_mean > 1000].index
mcds = mcds.sel({var_dim:use_features})
# mcds.add_mc_frac(var_dim=var_dim, normalize_per_cell=True, clip_norm_value=10)


In [ ]:
mcds

In [ ]:
mc_type = 'CHN'
adata = mcds.get_adata(mc_type=mc_type, var_dim=var_dim, select_hvf=False)


In [ ]:
adata.write_h5ad(f'{outdir}cell_{adata.shape[0]}_100kCH.h5ad')


In [ ]:
model = PCA(n_components=100, svd_solver='arpack', random_state=0)
adata.obsm['pca_all'] = model.fit_transform(adata.X)
npc = significant_pc_test(adata, obsm='pca_all', p_cutoff=0.01, update=False)


In [ ]:
npc = 100
adata.obsm['100kCH_pca'] = adata.obsm['pca_all'][:, :npc].copy()
tsne(adata, obsm='100kCH_pca', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
adata.obsm[f'100kCH_pc{npc}_tsne'] = adata.obsm['X_tsne'].copy()


In [ ]:
adata = adata[meta.index].copy()
adata.write_h5ad(f'{outdir}cell_{adata.shape[0]}_100kCH.h5ad')


In [ ]:
adata = anndata.read_h5ad(f'{outdir}cell_{meta.shape[0]}_100kCG.h5ad')


In [ ]:
adata = anndata.AnnData(obs=adata.obs, obsm=adata.obsm)
adata.write_h5ad(f'{outdir}cell_{adata.shape[0]}_100kCH_embed.h5ad')


In [ ]:
obs_dim = 'cell'
var_dim = 'chrom5k'


In [ ]:
mcds = MCDS.open(f'{indir}CEMBA.snmC.mcds', var_dim=var_dim)
mcds

In [ ]:
mcds = mcds.sel({obs_dim: mcds.get_index('cell').intersection(meta.index)})
mcds = mcds.remove_chromosome(exclude_chromosome=chrom_to_remove, var_dim=var_dim)
mcds = mcds.remove_black_list_region(black_list_path=black_list_path, f=0.5)


In [ ]:
mcad = mcds.get_score_adata(mc_type='CGN', 
                            quant_type='hypo-score',
                            loading_chunk=50000, 
                            binarize_cutoff=0.95
                           )

In [ ]:
filter_regions(mcad, n_cell=5)


In [ ]:
mcad.write_h5ad(f'{outdir}cell_{mcad.shape[0]}_5kCG.h5ad')


In [ ]:
mcad = anndata.read_h5ad(f'{outdir}cell_{meta.shape[0]}_5kCG.h5ad')


In [ ]:
model = LSI(scale_factor=10000,
            n_components=100,
            algorithm='arpack',
            random_state=0)


In [ ]:
model.fit(mcad, downsample=50000)
model.transform(mcad, obsm_name='5kCG_lsi')


In [ ]:
npc = significant_pc_test(mcad, p_cutoff=0.05, obsm='5kCG_lsi', update=False)


In [ ]:
mcad.write_h5ad(f'{outdir}cell_{mcad.shape[0]}_5kCG.h5ad')


In [ ]:
npc = 100
mcad.obsm['X_lsi'] = normalize(mcad.obsm['5kCG_lsi'][:, :npc], axis=1)
# sce.pp.harmony_integrate(mcad, 'Donor', basis='X_lsi', adjusted_basis=f'5kCG_u{npc}hm', 
#                          max_iter_harmony=30, random_state=0)
tsne(mcad, obsm='X_lsi', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'5kCG_u{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [ ]:
mcad = mcad[meta.index].copy()
mcad.write_h5ad(f'{outdir}cell_{mcad.shape[0]}_5kCG.h5ad')


In [ ]:
mcad = anndata.AnnData(obs=mcad.obs, obsm=mcad.obsm)
mcad.write_h5ad(f'{outdir}cell_{mcad.shape[0]}_5kCG_embed.h5ad')


In [ ]:
adata.obs = meta.copy()
mcad.obs = meta.copy()

In [ ]:
coord_base = 'tsne'
adata = dump_embedding(adata, coord_base)
mcad = dump_embedding(mcad, coord_base)


In [ ]:
ds = 0.05

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300, constrained_layout=True)

for i,tmp in enumerate([mcad, adata]):
    ax = axes[i]
    _ = categorical_scatter(data=tmp.obs,
                            ax=ax,
                            coord_base=coord_base,
                            hue='SubClass',
                            # text_anno='SubClass', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette=L2_color.to_dict(),
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            # show_legend=True
                           )

fig.savefig(f'{outdir}cemba_5kCG_100kCG_subclass.pdf', transparent=True)
